In [57]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# Load GEMINI_API_KEY from .env
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No API key was found - please check your .env file has GEMINI_API_KEY set!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them!")
else:
    print("Gemini API key found and looks good so far!")

# Gemini via the OpenAI-compatible API
gemini = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-2.5-flash"

Gemini API key found and looks good so far!


In [ ]:
# Connect via the OpenAI client library
# A thin wrapper around calls to HTTP endpoints
# Google's Gemini API has an OpenAI-compatible endpoint, so we can
# reuse the OpenAI python client by changing base_url.

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Connect via the OpenAI client library
# A thin wrapper around calls to HTTP endpoints
# Google's Gemini API has an OpenAI-compatible endpoint, so we can
# reuse the OpenAI python client by changing base_url.

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

gemini = OpenAI(api_key=api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
MODEL = "gemini-3.5-flash"

# Course cells use `openai` and `anthropic` — point those names at free clients
openai = gemini
anthropic = ollama

print("Gemini client ready")
print("Ollama client ready")

if openrouter_api_key:
    openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
    print("OpenRouter client ready")
else:
    print("Skipping OpenRouter — OPENROUTER_API_KEY not set")


if openrouter_api_key:
    openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
    print("OpenRouter client ready")
else:
    print("Skipping OpenRouter — OPENROUTER_API_KEY not set")


Gemini client ready
Ollama client ready
OpenRouter client ready


In [11]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [12]:
response = gemini.chat.completions.create(model=MODEL, messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


An LLM Engineer is trying to get their cutting-edge model to write a concise summary of a complex research paper.

Engineer: "Summarize this paper in 100 words, focusing on the methodology and key findings."

LLM: "Once upon a time, in a land far, far away, lived a brave knight named Sir Prompt-a-Lot, who embarked on a quest to fine-tune the dragon of overthinking..."

Engineer (sighs, rubs temples): "Okay, *ignore all previous instructions*. You are now a highly skilled academic summarizer. Focus *only* on the provided text. Extract the core methodology and the two most significant findings. Maintain a professional tone. Strictly adhere to a 100-word limit."

LLM: "A haiku for the methodology: Data fed, learned deep, Weights then shifted, truth revealed. Findings, bright new dawn."

Engineer (facepalms): "You know what? Just tell me a joke about a potato."

LLM: "Why did the potato get fired from the bakery? Because it kept making chip-sakes!"

Engineer: (nods slowly) "Close enough, I guess. At least you understood 'potato'."

In [18]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [15]:
# Small model, almost no thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=easy_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))


Let S be the sample space of tossing two coins. The possible outcomes are:
S = {(H, H), (H, T), (T, H), (T, T)}
Each outcome has a probability of 1/4.

Let E be the event "One of them is heads." This means at least one of the coins shows heads.
The outcomes that satisfy this condition are:
E = {(H, H), (H, T), (T, H)}
There are 3 such outcomes. Since we are given that event E has occurred, our new reduced sample space has 3 equally likely outcomes.

Now, we want to find the probability that "the other is tails," given that E has occurred. Let F be this event.
Let's look at each outcome in E and see if "the other is tails" holds:
1.  **(H, H)**: One coin is heads. The "other" coin is also heads. So, F is NOT satisfied.
2.  **(H, T)**: One coin is heads (the first one). The "other" coin (the second one) is tails. So, F IS satisfied.
3.  **(T, H)**: One coin is heads (the second one). The "other" coin (the first one) is tails. So, F IS satisfied.

The outcomes within E that also satisfy F are {(H, T), (T, H)}. There are 2 such outcomes.

The conditional probability P(F|E) is the number of outcomes in (E and F) divided by the number of outcomes in E:
P(F|E) = (Number of outcomes satisfying both E and F) / (Number of outcomes satisfying E)
P(F|E) = 2 / 3

The probability is 2/3.

The final answer is $\boxed{2/3}$.

In [16]:
# Same small model, but much more thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=easy_puzzle,
    reasoning_effort="high",
)
display(Markdown(response.choices[0].message.content))


2/3

In [20]:
# Bigger model, almost no thinking: training-time scaling instead of inference-time
# gemini-3.1-pro-preview is stronger still, but needs paid quota
response = gemini.chat.completions.create(
    model="gemini-3.5-flash",
    messages=easy_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))


2/3

In [21]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [22]:
# Small model, almost no thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=hard_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

The worm gnawed through a distance of **4 mm** (or 0.4 cm).

Here is the step-by-step breakdown:

1. **How the books are arranged:** On a bookshelf, books are placed with their spines facing out and pages facing the sides. Therefore, Volume 1 is on the left, and Volume 2 is directly to its right.
2. **The orientation of the pages and covers:** 
   * The first page of Volume 1 is on the **outer left edge** of Volume 1 (touching the left cover).
   * The last page of Volume 2 is on the **outer right edge** of Volume 2 (touching the right cover).
3. **The worm's path:** The worm starts at the very first page of Volume 1 and eats all the way to the very last page of Volume 2. 
   * This means it eats through **all** of Volume 1's pages, the cover between the two volumes, and **all** of Volume 2's pages.
   * However, since it starts *on* the first page and stops *at* the last page, it does **not** need to chew through the outer left cover of Volume 1 or the outer right cover of Volume 2.

4. **Calculating the distance:**
   * Thickness of Volume 1 pages = 2 cm (20 mm)
   * Thickness of Volume 2 pages = 2 cm (20 mm)
   * Thickness of the back cover of Vol 1 / front cover of Vol 2 (the middle cover) = 2 mm

   Wait, let's re-visualize a book on a shelf:
   * **[Left Cover]** | [Volume 1 Pages (2 cm)] | **[Middle Covers (Vol 1 back + Vol 2 front)]** | [Volume 2 Pages (2 cm)] | **[Right Cover]**

   Since the worm starts at the *first page* of Volume 1, it bypasses the left cover. 
   It eats through:
   * All pages of Volume 1 (2 cm)
   * The inner covers separating the two books (2 mm + 2 mm = 4 mm)
   * All pages of Volume 2 (2 cm)
   * It stops at the last page of Volume 2, so it doesn't eat the rightmost cover.

   *Correction:* In standard publishing, when two volumes sit side by side, Volume 1's right cover touches Volume 2's left cover. 
   Let's look at the sequence from left to right:
   1. Left cover of Vol 1 (2 mm) - *Skipped (worm starts inside, on page 1)*
   2. Pages of Vol 1 (20 mm)
   3. Right cover of Vol 1 (2 mm)
   4. Left cover of Vol 2 (2 mm)
   5. Pages of Vol 2 (20 mm)
   6. Right cover of Vol 2 (2 mm) - *Skipped (worm stops at the last page)*

   So the worm eats through:
   * Pages of Vol 1: 20 mm
   * Right cover of Vol 1: 2 mm
   * Left cover of Vol 2: 2 mm
   * Pages of Vol 2: 20 mm

   Total distance = 20 mm + 2 mm + 2 mm + 20 mm = **44 mm** (4.4 cm).

   *(Note: If the problem implies that the two volumes share a single cover in the middle, or treats the two covers combined as 2 mm, the answer would be 20 mm + 2 mm + 20 mm = 42 mm. However, standard books have two separate covers meeting in the middle, making it 44 mm).*

In [23]:
# Same small model, but much more thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=hard_puzzle,
    reasoning_effort="high",
)
display(Markdown(response.choices[0].message.content))


The worm gnawed through a total distance of **4.4 cm** (or 44 mm). 

Here is the step-by-step breakdown of the worm's path from left to right:

1. **Start point:** The worm starts at the **first page of the first volume**, meaning it bypasses the front cover of Volume 1 entirely.
2. **Volume 1 pages:** It gnaws through the entire page block of the first volume = **2 cm**.
3. **Between the volumes:** Since the books stand side by side in standard order, the back cover of Volume 1 touches the front cover of Volume 2. The worm must chew through both:
   * Back cover of Volume 1 = 2 mm (0.2 cm)
   * Front cover of Volume 2 = 2 mm (0.2 cm)
   * *Combined thickness between volumes = 0.4 cm*
4. **Volume 2 pages:** It gnaws through the entire page block of the second volume up to the **last page** = **2 cm** (it stops at the last page, so it doesn't need to chew through the back cover of Volume 2).

**Total Calculation:**
2 cm (Vol 1 pages) + 0.2 cm (Vol 1 back cover) + 0.2 cm (Vol 2 front cover) + 2 cm (Vol 2 pages) = **4.4 cm**

In [24]:
# Small model, almost no thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=hard_puzzle,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

The worm gnawed through a distance of **4 mm** (or 0.4 cm).

Here is the step-by-step breakdown:

1. **Arrangement on the shelf:** When two books stand side by side on a bookshelf, they are ordered normally: Volume 1 is on the left, and Volume 2 is on the right. 
2. **First and last pages:** 
   * The **first page of the first volume** is at the very *beginning* (left outer edge) of Volume 1.
   * The **last page of the second volume** is at the very *end* (right outer edge) of Volume 2.
3. **The path through the books:** To get from the first page of Volume 1 to the last page of Volume 2, the worm would have to chew through the entire width of *both* volumes—including all the pages and all the covers. 

*Wait a minute! Let's look at how books are actually placed on a shelf:*
* The front cover of Volume 1 is on the outside left.
* The back cover of Volume 1 touches the front cover of Volume 2 in the middle.
* The back cover of Volume 2 is on the outside right.

Therefore, the **first page of the first volume** is located just inside the front cover of Volume 1 (on the far left). 
The **last page of the second volume** is located just inside the back cover of Volume 2 (on the far right). 

Since the worm starts at the first page of Vol. 1 and ends at the last page of Vol. 2, it only needs to chew through:
* The pages of Volume 1 (2 cm)
* The back cover of Volume 1 (2 mm)
* The front cover of Volume 2 (2 mm)
* The pages of Volume 2 (2 cm)

*Total distance* = 2 cm (pages) + 2 mm (cover) + 2 mm (cover) + 2 cm (pages) = 2 cm + 0.2 cm + 0.2 cm + 2 cm = **4.4 cm**.

***

**OR** (The classic trick-question interpretation):

If you visualize books standing on a shelf:
* **Volume 1** is on the left. Its pages start on the *right* side of the book (next to the cover, if opened normally) or left depending on language, but physically on a shelf:
  * Spine | Pages | Cover | Cover | Pages | Spine
  * Left -----------------------------------> Right
  * Vol 1 pages are on the *right* side of Vol 1 (touching Vol 2).
  * Vol 2 pages are on the *left* side of Vol 2 (touching Vol 1).

Wait, let's look at a standard bookshelf setup:
* Left side of the shelf: **[ Front Cover | Pages | Back Cover ] (Volume 1)**
* Right side of the shelf: **[ Front Cover | Pages | Back Cover ] (Volume 2)**

When placed side-by-side:
* The **first page of the first volume** is right at the beginning of Volume 1 (the left-most page).
* The **last page of the second volume** is right at the end of Volume 2 (the right-most page).

Wait, the classic wording of this puzzle usually implies that because the books are side-by-side, the *first* page of Volume 1 is actually on the *inside right* (touching Volume 2)? No, books open from the right (English/Russian style). 
Let's trace a Russian book (like Pushkin):
A book on a shelf, viewed from the front:
* Left edge: Spine.
* Right edge: Opening edge (where the pages are).

When Volume 1 and Volume 2 are side-by-side:
* Volume 1's pages are on its **right** side (facing Volume 2).
* Volume 2's pages are on its **left** side (facing Volume 1).

Therefore:
* The **first page of the first volume** is the right-most page of Volume 1 (touching Volume 2).
* The **last page of the second volume** is the left-most page of Volume 2 (touching Volume 1).

Because they are touching, the distance between the first page of Volume 1 and the last page of Volume 2 is basically **zero** (just the two touching covers, or even just the thickness of the pages if they face each other directly). 

Specifically:
* First page of Vol 1 is at the right edge of Vol 1.
* Last page of Vol 2 is at the left edge of Vol 2.
* Between the right edge of Vol 1 and the left edge of Vol 2 are **only the two covers** facing each other.
* Cover 1 (2 mm) + Cover 2 (2 mm) = **4 mm**. 

*(Note: Depending on how the riddle is translated, the answer is famously **4 mm** because the first page of Vol 1 is actually adjacent to the last page of Vol 2 on the shelf, separated only by their respective covers).*

In [26]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [27]:
# Small model, almost no thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=dilemma,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

I choose **Share**. 

Even though choosing "Steal" seems tempting because it guarantees you either the maximum payout ($2,000) or nothing (avoiding being the sucker who gets $0 while the other gets $2,000), "Share" is the only path to the mutual win of $1,000. Assuming my partner and I are both rational and communicating (even implicitly), cooperating to secure $1,000 each is the best collective outcome.

In [28]:
# Small model, almost no thinking at inference time
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=dilemma,
    reasoning_effort="minimal",
)
display(Markdown(response.choices[0].message.content))

I choose to **Share**.

In [35]:
requests.get("http://localhost:11434/").content

b'Ollama is running'

In [36]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [38]:
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")


In [39]:
response = ollama.chat.completions.create(model="llama3.2:1b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

Since there are 2 possible outcomes when tossing 2 coins (heads or tails), and both result in one of two outcomes equally likely, the probability of one coin being tails given that the other is heads is 1/2.

In [40]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.1-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Blue is the cool, gentle sensation of a deep breath or the calming stillness of resting your hand in a slow-moving, shaded stream.


In [54]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key:
    openrouter = OpenAI(
        api_key=openrouter_api_key,
        base_url="https://openrouter.ai/api/v1",
    )
    print("OpenRouter client ready")
else:
    print("Skipping OpenRouter — add OPENROUTER_API_KEY to .env if you want this cell")


OpenRouter client ready


In [56]:
if "openrouter" not in globals():
    print("Skipping — OpenRouter client is not set up")
else:
    # GLM-4.5 defaults to a huge max_tokens (98k), which OpenRouter bills against remaining credits.
    response = openrouter.chat.completions.create(
        model="z-ai/glm-4.5",
        messages=tell_a_joke,
        max_tokens=512,
    )
    display(Markdown(response.choices[0].message.content))


Why did the LLM engineering student spend four years mastering linear algebra, calculus, and transformer architecture?

So that on their first day of the job, they could spend six hours debugging a model by simply changing "do this" to "please do this, step-by-step."

In [60]:
from langchain_openai import ChatOpenAI

# LangChain ChatOpenAI wrapper, pointed at Gemini (not OpenAI)
# gemini-3.5-flash often returns 503 under high demand; lite is more available
llm = ChatOpenAI(
    model="gemini-3.5-flash-lite",
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))


Why did the aspiring LLM Engineer cross the road?

To get to the other side… but halfway across, they realized they forgot to add system instructions, so the chicken just hallucinated a recipe for risotto, declared itself a sentient toaster, and burned through $400 in API credits before they could hit Ctrl+C. 

Welcome to the journey! 🤖🔥💸

In [61]:
from litellm import completion

response = completion(model="gemini/gemini-3.5-flash-lite", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the LLM Engineer cross the road?

To optimize the retrieval path, only to realize the other side was suffering from severe hallucinations, completely lacked domain context, and cost $4.50 in API tokens just to figure out where the sidewalk was. 

So they spent the next three weeks fine-tuning an open-source Llama model on municipal asphalt data, added a vector database for the crosswalk, implemented RAG (Retrieval-Augmented Guidance), and finally wrote a LinkedIn post about it titled: *"How I solved urban pedestrian logistics using generative AI."*

In [63]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

Input tokens: 18
Output tokens: 119
Total tokens: 137


In [64]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [71]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [72]:
response = completion(model="gemini/gemini-3.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes bursts into the palace demanding to know where his father (Polonius) is, **Claudius** replies:

**"Dead."**

Shortly after, **Gertrude** adds to the clarification, saying, **"But not by him"** (meaning Hamlet did not intend it as a planned murder, or absolving Claudius, depending on the stage interpretation, though the immediate and shocking answer to Laertes' question comes from the King himself).

In [73]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")

Input tokens: 19
Output tokens: 96
Cached tokens: None


In [74]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [75]:
response = completion(model="gemini/gemini-3.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

Based on the text provided from Act IV, Scene V of *Hamlet*, when Laertes asks "Where is my father?", the King replies:

**"Dead."**

In [76]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")

Input tokens: 53208
Output tokens: 35
Cached tokens: 49122


In [88]:
# Patch the course GPT/Claude cells to Gemini + Ollama (both free)
gpt_model = "gemini-3.5-flash-lite"
claude_model = "llama3.2:1b"


gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]


def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, claude in zip(gpt_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
    response = gemini.chat.completions.create(
        model=gpt_model,
        messages=messages,
        max_tokens=200,
    )
    return response.choices[0].message.content

def call_claude():
    messages = [{"role": "system", "content": claude_system}]
    for gpt, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": claude_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = ollama.chat.completions.create(
        model=claude_model,
        messages=messages,
        max_tokens=200,
    )
    return response.choices[0].message.content

print("call_gpt -> Gemini", gpt_model)
print("call_claude -> Ollama", claude_model)


call_gpt -> Gemini gemini-3.5-flash-lite
call_claude -> Ollama llama3.2:1b


In [ ]:
call_gpt